In [ ]:
# 1. Install Unsloth (The core engine)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install the Hugging Face Ecosystem
!pip install transformers datasets trl peft accelerate bitsandbytes

# 3. Install xformers (Memory optimization)
!pip install xformers

In [ ]:
import torch
from unsloth import FastLanguageModel

# 1. Load the Base Model in 4-bit Quantization
print("Loading Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048, # How much context the model can remember at once
    dtype = None, # Auto-detects the best format for your GPU
    load_in_4bit = True, # Shrinks the model to fit in the free GPU
)

# 2. Add the LoRA Adapters (This is the part we will actually train)
print("Applying LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Size of the adapter
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Connecting to all attention layers
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
)

print(f"\n✅ SUCCESS! Connected to GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Model Llama-3.2-3B is locked, loaded, and ready to fine-tune!")

In [ ]:
from datasets import load_dataset

# 1. Define the exact structure the model will learn to read
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 2. Add the "Stop Talking" token (Crucial!)
EOS_TOKEN = tokenizer.eos_token # Without this, the model will generate endless gibberish

# 3. Create the formatting function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input, output in zip(instructions, inputs, outputs):
        # We inject the data into our template and cap it with the EOS token
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 4. Download and process the data
print("Downloading dataset...")
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")

# Here we will only use the first 2,000 rows to save time
dataset = dataset.select(range(2000))

print("Formatting data...")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. Sanity Check: Let's look at the very first row
print("\n✅ Dataset Ready! Here is exactly what the model will see for row 1:\n")
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configuring the Trainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # We are doing a fast sprint of 60 steps!
        learning_rate = 2e-4, # Standard learning rate for LoRA
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Uses less memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Disabling WandB logging for this quick sprint
    ),
)

print("Starting the Engine. Training commencing now...\n")
trainer_stats = trainer.train()

print(f"\n✅ TRAINING COMPLETE!")

In [ ]:
from transformers import TextStreamer

# 1. Format your prompt
prompt = alpaca_prompt.format(
    "Explain the concept of smartphone in one simple paragraph.", # Instruction
    "", # Input
    "", # Response (Left blank)
)

# 2. Tokenize and send to GPU
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Set up the Streamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)

# 4. Generate the output!
print("🧠 Model is thinking...\n")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    pad_token_id = tokenizer.eos_token_id,
    use_cache = False # <--- THE FIX: Bypassing the KV-Cache bug
)

In [ ]:
from huggingface_hub import login
login() # Paste your 'Write' token here when prompted


In [ ]:
# Replace 'your-username' with your actual Hugging Face username
model.push_to_hub("kumarsarthak98/llama-3.2-3b-agentic-sprint", token = True)
tokenizer.push_to_hub("kumarsarthak98/llama-3.2-3b-agentic-sprint", token = True)

print("🚀 Model is live on the Hub!")

In [ ]:
!pip install gradio
import gradio as gr

In [ ]:
def generate_response(instruction, user_input=""):
    # 1. Format using the Alpaca template we used for training
    prompt = alpaca_prompt.format(instruction, user_input, "")

    # 2. Tokenize
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # 3. Generate (using the use_cache=False fix we found earlier)
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    # 4. Decode and clean up the output
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

    # Remove the prompt part and only return the "Response"
    response = decoded.split("### Response:")[-1].strip()
    return response

In [ ]:
import gradio as gr

def generate_response(instruction, user_input=""):
    prompt = alpaca_prompt.format(instruction, user_input, "")
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # We use a shorter max_new_tokens for the demo to keep it snappy
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256, # Increased this so you get longer answers
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response = decoded.split("### Response:")[-1].strip()
    return response

# --- IMPROVED UI LAYOUT ---
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🤖 My Agentic AI Fine-Tune")
    gr.Markdown("Fine-tuned Llama-3.2-3B. Optimized for instruction following.")

    with gr.Row():
        with gr.Column():
            instr = gr.Textbox(
                label="Instruction",
                placeholder="What should the AI do?",
                lines=3 # Makes the input box taller
            )
            ctx = gr.Textbox(
                label="Input/Context (Optional)",
                placeholder="Paste background data here...",
                lines=5
            )
            submit_btn = gr.Button("Generate Response", variant="primary")

        with gr.Column():
            # THIS IS THE BOX YOU WANTED TO INCREASE
            out = gr.Textbox(
                label="AI Response",
                lines=15,       # Sets the initial height to 15 lines
                max_lines=30,   # Allows it to grow up to 30 lines before scrolling
                interactive=False,
                show_copy_button=True # Great for user experience
            )

    # Link the button to the function
    submit_btn.click(fn=generate_response, inputs=[instr, ctx], outputs=out)

demo.launch(share=True)

In [ ]:
%%writefile app.py
import gradio as gr
import torch
from unsloth import FastLanguageModel

# 1. Load the model and tokenizer (use the same settings as your sprint)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Define the prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 3. Define the generation function
def generate_response(instruction, user_input=""):
    prompt = alpaca_prompt.format(instruction, user_input, "")
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response = decoded.split("### Response:")[-1].strip()
    return response

# 4. Build the Gradio Interface
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🤖 My Agentic AI Fine-Tune")
    with gr.Row():
        with gr.Column():
            instr = gr.Textbox(label="Instruction", lines=3)
            ctx = gr.Textbox(label="Input/Context (Optional)", lines=5)
            submit_btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            out = gr.Textbox(label="AI Response", lines=15)

    submit_btn.click(fn=generate_response, inputs=[instr, ctx], outputs=out)

# 5. Launch (Note: No 'share=True' needed for HF Spaces)
demo.launch()